# Scouting

Notebooks for in-season analysis of stats.


What's needed?

[] Get current CBS rosters and free agents. (Hitters, pitchers)

[] Get current, helpful stats and projects

[] Merge these.

[] Scout!

In [2]:
import os
import sys
import pandas as pd
import numpy as np
import functools

sys.path.append(r'/home/lenhart/Repos/phi-utils')

from philosofool.data_sources.utils import read_yml
from fantasy_baseball_draft.utils import StatSynonyms, load_cbs_data, DataLoader
from fantasy_baseball_draft import spg


/tmp/ipykernel_2153951/900476029.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [3]:
def filter_avail(df, regex):
    return df[df.Avail.str.contains(regex)]

def filter_elig(df, position):
    return df[matches_eligible(df.Eligible, position)]

def filter_name(df, regex):
    return df[df.Player.str.contains(regex, re.IGNORECASE)]

In [4]:
data_path = read_yml('local/config.yml')['paths']['local_data']
scouting_path = os.path.join(data_path, 'scouting')
loader = DataLoader(scouting_path)
recent_pitchers = 'pitchers_tt_4-19-24.csv'

In [5]:
FA_RE = 'FA|W\s?\('
OMAK_RE = 'Omak'

In [6]:
pitchers = loader.load_cbs_csv(recent_pitchers).rename(columns={'BFP': 'PA'}).assign(k_100=lambda df: df.K / df.PC * 100)

In [7]:
pitchers.query('PA > 30').sort_values('k_100', ascending=False).pipe(filter_avail, 'Czar').head(10)

,Avail,Player,IP,GS,PA,PC,W,S,K,BB,H,HR,ER,ERA,WHIP,Kd9,BBd9,Rank,k_100
174,Cackleberry Czars,Aaron Civale P | TB,17.0,3,66,251,2,0,18,4,12,3,4,2.12,0.94,9.53,2.12,34,7.171315
236,Cackleberry Czars,David Robertson P | TEX,8.3,0,31,113,1,0,8,3,4,1,1,1.08,0.84,8.64,3.24,142,7.079646
170,Cackleberry Czars,Spencer Turnbull P | PHI,15.0,3,61,253,1,0,16,5,9,1,3,1.80,0.93,9.60,3.00,66,6.324111
146,Cackleberry Czars,Garrett Whitlock P | BOS,14.3,3,59,267,1,0,16,6,11,1,2,1.26,1.19,10.05,3.77,90,5.992509
321,Cackleberry Czars,Frankie Montas P | CIN,16.7,3,68,277,2,0,13,4,15,1,4,2.16,1.14,7.02,2.16,63,4.693141


In [33]:
pitchers.query('GS > -1').sort_values('k_100', ascending=False).pipe(filter_avail, 'Omak').head(10)

,Avail,Player,IP,GS,PA,PC,W,S,K,BB,H,HR,ER,ERA,WHIP,Kd9,BBd9,Rank,k_100
26,Omak Wrong Players,Pete Fairbanks P | TB,5.0,0,26,113,0,3,8,6,5,0,5,9.00,2.20,14.40,10.80,526,7.079646
14,Omak Wrong Players,Blake Snell P | SF,3.0,1,14,72,0,0,5,2,3,0,3,9.00,1.67,15.00,6.00,614,6.944444
164,Omak Wrong Players,Tanner Houck P | BOS,17.7,3,76,280,2,0,19,2,19,1,4,2.04,1.19,9.68,1.02,50,6.785714
100,Omak Wrong Players,Sean Manaea P | NYM,14.7,3,65,269,1,0,18,7,13,1,7,4.29,1.36,11.05,4.30,245,6.691450
82,Omak Wrong Players,Ryne Stanek P | SEA,5.3,0,25,106,0,2,7,3,5,1,2,3.38,1.50,11.81,5.06,345,6.603774
262,Omak Wrong Players,Reynaldo Lopez P | ATL,12.0,2,45,176,1,0,11,5,7,0,1,0.75,1.00,8.25,3.75,94,6.250000
273,Omak Wrong Players,Tanner Scott P | MIA,6.7,0,33,137,0,1,6,9,2,0,1,1.35,1.65,8.10,12.15,398,4.379562
427,Omak Wrong Players,Triston McKenzie P | CLE,13.0,3,62,249,1,0,5,12,11,2,9,6.23,1.77,3.46,8.31,692,2.008032
1649,Omak Wrong Players,Kodai Senga P | NYM,0.0,0,0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,9999,NaN


In [ ]:
def most_recently_modified(path:str, exp: str) -> str:
    files = [(f, os.path.getmtime(os.path.join(path, f))) for f in os.listdir(path) if re.search(exp, f)]
    return sorted(files, key=lambda x: x[1])[-1][0]

most_recently_modified(data_path, 'hitter.*csv')

In [ ]:
def merge_fwar(df, fwar_df):
    #fwar_df = fwar_df[['Player', 'playerid', 'fwar']]
    df = df[['Player', 'Avail'] + [col for col in df if col not in fwar_df.columns]]
    df = fwar_df.drop(['Avail'], axis=1).merge(df, on='Player', how='left')
    first = ['Avail', 'Player']
    df = df[first + [col for col in df.columns if col not in first]]
    return df.sort_values('fwar', ascending=False)

data_path = os.path.join(read_yml('local/config.yml')['paths']['local_data'], 'in_season')
loader = DataLoader(data_path)

hitters = merge_fwar(
    loader.load_cbs_csv(most_recently_modified(data_path, 'hitter.*.csv')),
    pd.read_csv('local/hitter_proj.csv', index_col='index'))

pitchers = merge_fwar(
    loader.load_cbs_csv(most_recently_modified(data_path, 'pitcher.*.csv')),
    pd.read_csv('local/pitcher_proj.csv', index_col='index'))


spgs = spg.spgs_from_standings_html('/home/lenhart/Dropbox/baseball/fantasy_data/standings/cbs_2021_standings.html')
rate_valuator = spg.FantasyRateValuator(spgs)

hitters['rwar'] = rate_valuator.hitter_fwar(hitters) - 7.37
pitchers['rwar'] = rate_valuator.pitcher_fwar(pitchers) - 4.84

In [ ]:
def get_team(df, team='Omak'):
    return df.dropna()[df.dropna().Avail.str.contains(team)]

def get_position(df, positions):
    pos = []
    for p in positions.split(','):
        pos.append(f'{p},|{p}$')
    pos = '|'.join(pos)
    return df[df.Eligible.str.match(pos)]

def get_free_agents(df):
    team = r"(W(\s?)\()|(FA)"
    return get_team(df, team)

get_team(pitchers).round(2)

In [ ]:
get_team(hitters).drop(columns=['Rank','Avail']).round(1)

In [ ]:
get_free_agents(hitters.sort_values('rwar', ascending=False)).head(10).round(3)

In [ ]:
get_free_agents(pitchers.sort_values('fwar', ascending=False)).head(10)

In [ ]:
get_free_agents(get_position(hitters, '3B')).drop(columns=['Rank']).round(3).head(10)

## YTD Performance

Let's look for existing outliers.

In [ ]:
1.2 * 1.1 * 1.1

In [ ]:


def new_metrics(df):
    df = df.copy()
    df['k100'] = (df.K / df.PC *100).replace(np.inf, np.nan).fillna(0.)
    return df

pitchers = (loader.load_cbs_csv(most_recently_modified(data_path, 'pitcher.*csv'))
    .pipe(new_metrics)
    .pipe(lambda df: df[df.IP > 0])
)

pitchers[free_agents(pitchers)].sort_values('k100').tail(10)

## Trades

In [ ]:
offered = ['ac Par', 'sco Alv']
requested = ['Snell ']
chips = '|'.join(offered + requested)

def trade_comparison(pitchers, hitters, chips) -> pd.DataFrame:
    def comparison(df):
        omak = get_team(df)
        df = df.drop(omak.index)
        return pd.concat([
            df[df.Player.str.contains(chips)],
            omak[omak.Player.str.contains(chips)]
        ])
    cols = ['Avail', 'Player', 'IP', 'W', 'S', 'K', 'BB', 'ERA', 'WHIP', 'AB','R', 'HR', 'RBI', 'SB', 'BA', 'OBP', 'SLG', 'playerid', 'fwar']
    df = pd.concat([comparison(pitchers), comparison(hitters)]).fillna(0.)[cols]
    return df

compare = functools.partial(trade_comparison, pitchers, hitters)

result = compare(chips)
balance = np.where(result.Avail.str.contains('Omak'), result.fwar * -1, result.fwar).sum()
print(balance)
pd.DataFrame.round(result.copy(), 10)
result.round(2)

In [ ]:
hitters[hitters.Player.str.contains(offered)]

In [ ]:
pitchers[free_agents(pitchers)]
pitchers['k9'] = pitchers.K / pitchers.IP * 9
pitchers['bb9'] = pitchers.BB / pitchers.IP * 9
pitchers.sort_values('k9').dropna().tail(20)

In [ ]:
pitchers